# Overview

This notebook is used to manually test the tools before refactoring the code to work with `LangGraph` in the `tools/custom_tools` folder.

In [0]:
%pip install python-dotenv twilio


In [0]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  

AWS_REGION      = os.getenv("AWS_REGION")
AWS_ACCESS_KEY  = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY  = os.getenv("AWS_SECRET_KEY")
SENDER          = os.getenv("SENDER")
RECIPIENT       = os.getenv("RECIPIENT")
SMS_ACCOUNT_SID = os.getenv("SMS_ACCOUNT_SID")
SMS_AUTH_TOKEN  = os.getenv("SMS_AUTH_TOKEN")

def loaded(val):
    return "✅ loaded" if val else "❌ not loaded"

print("AWS_REGION:", loaded(AWS_REGION))
print("AWS_ACCESS_KEY:", loaded(AWS_ACCESS_KEY))
print("AWS_SECRET_KEY:", loaded(AWS_SECRET_KEY))
print("SENDER:", loaded(SENDER))
print("RECIPIENT:", loaded(RECIPIENT))
print("SMS_ACCOUNT_SID:", loaded(SMS_ACCOUNT_SID))
print("SMS_AUTH_TOKEN:", loaded(SMS_AUTH_TOKEN))

## Test Twilio

In [0]:
from twilio.rest import Client

# Find your Account SID and Auth Token at twilio.com/console
# and set the environment variables. See http://twil.io/secure

client = Client(SMS_ACCOUNT_SID, SMS_AUTH_TOKEN)

message = client.messages.create(
    body="this is a test sms",
    from_="+18773559479",
    to="3019085817",
)

print(message.body)

## Test Amazon SES (Simple Email Service)

In [0]:
import os, re, boto3

PALETTE = {"bg":"#f5f7fb","ink":"#222","red":"#e24a4a","blue":"#1e63c6","div":"#e6edf6"}

def send_email(recipient: str, subject: str, message: str):
    """Send a styled HTML email via AWS SES."""
    try:
        text = re.sub(r"<[^>]+>", "", message.replace("<br/>", "\n").replace("<br>", "\n"))
        html = f"""<html><body style="margin:0;padding:0;background:{PALETTE['bg']}">
        <table width="100%"><tr><td align="center" style="padding:24px">
        <table width="640" style="max-width:640px;background:#fff;border-radius:14px;box-shadow:0 6px 18px rgba(0,0,0,.06)">
        <tr><td style="padding:28px 26px;font-family:Segoe UI,Roboto,Arial,sans-serif;color:{PALETTE['ink']};font-size:16px">
        <div style="font-size:30px;font-weight:800;color:{PALETTE['red']}">Jackson &amp; Jackson</div>
        <span style="display:inline-block;background:{PALETTE['blue']};color:#fff;padding:4px 10px;border-radius:999px;font-size:12px;font-weight:700;margin:6px 0 14px">UPDATE</span>
        <hr style="border:none;border-top:1px solid {PALETTE['div']};margin:10px 0 18px">
        <div>{message}</div>
        <hr style="border:none;border-top:1px solid {PALETTE['div']};margin:24px 0 12px">
        <p style="font-size:12px;color:#667;text-align:center">© Jackson &amp; Jackson • Sent via AWS SES •
        <a href="mailto:{SENDER}" style="color:{PALETTE['blue']};text-decoration:none">{SENDER}</a></p>
        </td></tr></table></td></tr></table></body></html>"""

        ses = boto3.client(
            "ses",
            region_name=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY,
            aws_secret_access_key=AWS_SECRET_KEY,
        )

        resp = ses.send_email(
            Source=SENDER,
            Destination={"ToAddresses": [recipient]},
            Message={
                "Subject": {"Data": subject},
                "Body": {"Html": {"Data": html}, "Text": {"Data": text}},
            },
        )
        print(f"✅ Sent to {recipient}\nMessageId: {resp['MessageId']}")
    except Exception as e:
        print(f"❌ {type(e).__name__}: {e}")

# --- Run a test email ---
send_email(
    recipient=RECIPIENT,
    subject="SES Email Test",
    message="<p>Hello from Databricks — this is a test email via <b>AWS SES</b>.</p>"
)

## Test Mailgun

In [0]:
import os
import requests

def send_simple_message():
    response = requests.post(
        "https://api.mailgun.net/v3/datakafe.com/messages",
        auth=("api", os.getenv('API_KEY', '991a5d50c1a4d42fc8065d1bf13937d6-556e0aa9-858a86ec')),
        data={
            "from": "AI AGENT <robert@datakafe.com>",
            "to": "robert <robert@datakafe.com>",
            "subject": "Hello robert",
            "text": "HIIII robert, you just sent an email with Mailgun! You are truly awesome!"
        }
    )
    return response

# Invoke the function and print the response object
response = send_simple_message()
display(response)